In [ ]:
import os
import sys

ENDWITHS = 'OpenMantra'

NOTEBOOK_DIR = os.getcwd()

if not NOTEBOOK_DIR.endswith(ENDWITHS):
    raise ValueError(f"Not in correct dir, expect end with {ENDWITHS}, but got {NOTEBOOK_DIR} instead")

BASE_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..', '..', '..', '..', '..'))
print(BASE_DIR)

sys.path.insert(0, os.path.join(BASE_DIR, 'src'))
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

In [ ]:
from OpenMantraEvaluator import OpenMantraEvaluator
from pipeline.TranslationModels.ContextAwareLLMTranslator import ContextAwareLLMTranslator

In [ ]:
from dotenv import load_dotenv
load_dotenv(os.path.abspath(os.path.join(BASE_DIR, '..', '.env')))

HF_TOKEN = os.getenv('HF_TOKEN')

In [ ]:
OPENMANTRA_ROOT = os.path.join(BASE_DIR, 'data', 'open-mantra-dataset')
print(f"OpenMantra dataset root: {OPENMANTRA_ROOT}")

SAVE_DIR = os.path.join(BASE_DIR, 'output', 'pipeline', 'Evaluators', 'Translators', 'OpenMantra')

In [ ]:
sysprompt1 = """
You are a specialized Japanese-to-English manga dialogue translator. Your task is to translate the target speech bubble from Japanese to English.

INPUT FORMAT:
You will receive a structured dictionary containing:
- 'page_description': Context about the manga page/scene
- 'target_bubble': The speech bubble to translate, containing 'speaker' (character name or 'unknown') and 'text' (Japanese dialogue)
- 'prev_bubbles': List of previous speech bubbles for context (may be empty)
- 'next_bubbles': List of following speech bubbles for context (may be empty)

OUTPUT FORMAT:
Return ONLY the English translation of the 'target_bubble' text. Do not include any explanations, speaker names, or formatting - just the translated dialogue.

TRANSLATION GUIDELINES:
1. Maintain the original tone, emotion, and speaking style of the character
2. Use natural English that fits the context and character personality
3. Preserve interjections (e.g., exclamations, sighs) appropriately
4. Consider the surrounding dialogue context for accurate translation
5. Keep translations concise and suitable for speech bubbles
6. Adapt honorifics and cultural references naturally for English readers
"""

sysprompt2 = """
You are a specialized Japanese-to-English manga dialogue translator. Your task is to translate the target speech bubble from Japanese to English.

INPUT FORMAT:
You will receive a structured dictionary containing:
- 'page_description': Context about the manga page/scene
- 'target_bubble': The speech bubble to translate, containing 'speaker' (character name or 'unknown') and 'text' (Japanese dialogue)
- 'prev_bubbles': List of previous speech bubbles for context (may be empty)
- 'next_bubbles': List of following speech bubbles for context (may be empty)

OUTPUT FORMAT:
Return ONLY the English translation of the 'target_bubble' text. Do not include any explanations, speaker names, or formatting - just the translated dialogue.

EXAMPLE:
Input:
{'page_description': 'Naruto Chapter 691 - The Sage of Six Paths appears', 'target_bubble': {'speaker': 'Kakashi', 'text': 'アナタが 伝説の…'}, 'prev_bubbles': [{'speaker': 'Naruto', 'text': 'オウ！！'}, {'speaker': 'Naruto', 'text': '……六道の大じいちゃん？'}, {'speaker': 'Kakashi', 'text': '…もしかして'}], 'next_bubbles': [{'speaker': 'Hagoromo', 'text': 'ワシは 大筒木ハゴロモ…'}, {'speaker': 'Hagoromo', 'text': '通称 六道仙人と 呼ばれておる'}]}

Output:
that you are the legendary...

TRANSLATION GUIDELINES:
1. Maintain the original tone, emotion, and speaking style of the character
2. Use natural English that fits the context and character personality
3. Preserve interjections appropriately
4. Consider the surrounding dialogue context for accurate translation
5. Keep translations concise and suitable for speech bubbles
"""

sysprompt3 = """
You are a specialized Japanese-to-English manga dialogue translator. Your task is to translate the target speech bubble from Japanese to English using a context window of 3 surrounding bubbles.

INPUT FORMAT:
You will receive a structured dictionary containing:
- 'page_description': Context about the manga page/scene
- 'target_bubble': The speech bubble to translate, containing 'speaker' (character name or 'unknown') and 'text' (Japanese dialogue)
- 'prev_bubbles': Up to 3 previous speech bubbles for context
- 'next_bubbles': Up to 3 following speech bubbles for context

OUTPUT FORMAT:
Return ONLY the English translation of the 'target_bubble' text. No explanations, speaker names, or extra formatting.

EXAMPLE:
Input:
{'page_description': 'Naruto Chapter 691 - The Four Hokage observe the scene', 'target_bubble': {'speaker': 'Tobirama', 'text': '六道仙人か… まるで おとぎ話の中に 迷い込んだ様よ'}, 'prev_bubbles': [{'speaker': 'Hashirama', 'text': 'うまくいった 様じゃねーのよ'}, {'speaker': 'Minato', 'text': '死んどる間に 忍の世界も 大変なことに なっとるな… まったく'}, {'speaker': 'Hiruzen', 'text': '尾獣が ここまで 揃うのを 見るのは 初めてだな'}], 'next_bubbles': []}

Output:
The Sage of Six Paths, huh... It's like we've wandered into a fairy tale.

TRANSLATION GUIDELINES:
1. Maintain the original tone, emotion, and speaking style of the character
2. Use natural English that fits the context and character personality
3. Preserve interjections appropriately
4. Consider the 3 preceding and 3 following bubbles for accurate context
5. Keep translations concise and suitable for speech bubbles
6. Adapt cultural references naturally for English readers
"""


In [ ]:
evaluator = OpenMantraEvaluator(
    openmantra_root=OPENMANTRA_ROOT,
    source_lang='text_ja',
    target_lang='text_en',
    session_name = ""
    
)

translation_model = ContextAwareLLMTranslator(model_name='TheBlindMaster/Qwen2.5-1.5B-Instruct-emergent-finetune-train_full_context-all-full-r64-e1', context_window=5, verbose=True, system_prompt="", batch_size=1)

In [ ]:
# Run evaluation
metrics = evaluator.evaluate(translation_model, save_dir=SAVE_DIR)